# CMU 350 — Data Scraping: de una página web a evidencia
**Pregunta guía:** ¿Cómo convertimos información diseñada para personas en datos que una computadora puede analizar?

**WEB → HTML → SCRAPER → RAW DATA → CLEAN DATA → ANALYSIS → EVIDENCE**

> Notebook para una clase de **80 minutos**. Detente en las preguntas **PREDICE / DISCUTE**.

## 1 · Mira primero como humano — 0–8 min
Abre `agenda-cultural-pr.html`.

**PREDICE / DISCUTE**
1. ¿Qué variables reconoces?
2. ¿Qué pregunta podríamos contestar si tuviéramos todos los eventos en una tabla?
3. ¿Qué información parece numérica pero quizá no lo sea?

## 2 · ¿Qué ve la computadora? — 8–17 min
Inspecciona un evento en el navegador.

```html
<article class="event-card">
  <h2 class="title">Jazz bajo las estrellas</h2>
  <span class="venue">Museo de Arte de Puerto Rico</span>
  <span class="price">Gratis</span>
</article>
```
**Idea clave:** HTML contiene estructura; un *selector* indica qué parte queremos.

In [ ]:
from pathlib import Path
from bs4 import BeautifulSoup
import pandas as pd
import matplotlib.pyplot as plt

html = Path("agenda-cultural-pr.html").read_text(encoding="utf-8")
soup = BeautifulSoup(html, "html.parser")
cards = soup.select(".event-card")
len(cards)

### Alto
¿Qué son esos 32 objetos? ¿Por qué `.event-card` los encuentra?

In [ ]:
card = cards[0]
print(card.prettify()[:1200])

## 3 · Primer scrape — 17–27 min
**PREDICE:** si `.title` identifica el título, ¿qué selector usarías para el precio?

In [ ]:
title = card.select_one(".title").get_text(strip=True)
price = card.select_one(".price").get_text(strip=True)
title, price

In [ ]:
venue = card.select_one(".venue").get_text(strip=True)
municipality = card.select_one(".municipality").get_text(strip=True)
category = card.select_one(".category").get_text(strip=True)
venue, municipality, category

## 4 · Página → dataset — 27–39 min
Ya sabemos extraer **un** evento. Repetimos la regla.

In [ ]:
events = []
for card in cards:
    events.append({
        "title": card.select_one(".title").get_text(strip=True),
        "date": card.select_one(".date").get_text(strip=True),
        "venue": card.select_one(".venue").get_text(strip=True),
        "municipality": card.select_one(".municipality").get_text(strip=True),
        "category": card.select_one(".category").get_text(strip=True),
        "price": card.select_one(".price").get_text(strip=True)
    })

df = pd.DataFrame(events)
df.head(10)

In [ ]:
df.shape

### DISCUTE
¿En qué momento esto se convirtió en un dataset?

¿El scraper *descubrió* las variables o nosotros decidimos cuáles eran?

## 5 · La web no entrega datos limpios — 39–51 min
Primero mira los precios **sin corregirlos**.

In [ ]:
df["price"].value_counts(dropna=False)

Encuentra los problemas antes de programar.

`Gratis` · `FREE` · `$10` · `$10.00` · `$15-25` · precio ausente

**Decisión para hoy:** `Gratis/FREE → 0`; rango → punto medio; ausente → `NaN`.

In [ ]:
def clean_price(value):
    value = value.strip()
    if value.lower() in ["gratis", "free"]:
        return 0.0
    if value == "":
        return None
    value = value.replace("$", "")
    if "-" in value:
        low, high = value.split("-")
        return (float(low) + float(high)) / 2
    return float(value)

df["price_numeric"] = df["price"].apply(clean_price)
df["is_free"] = df["price_numeric"].eq(0)
df[["title","price","price_numeric","is_free"]].head(12)

### Otro problema escondido: ¿ves dos nombres para el mismo venue?

In [ ]:
sorted(df["venue"].unique())

In [ ]:
df["venue_clean"] = df["venue"].replace({
    "Museo de Arte PR": "Museo de Arte de Puerto Rico"
})
df["venue_clean"].value_counts().head()

## 6 · Investigación — 51–64 min
La clase escoge **UNA**:

**A.** ¿Qué tipos de eventos son más comunes?  
**B.** ¿Qué municipios concentran más eventos?  
**C.** ¿Qué categorías parecen ofrecer más eventos gratuitos?

In [ ]:
# A
counts = df["category"].value_counts()
display(counts)
counts.plot(kind="bar", title="Eventos por categoría")
plt.ylabel("Número de eventos")
plt.show()

In [ ]:
# B
counts = df["municipality"].value_counts()
display(counts)
counts.plot(kind="bar", title="Eventos por municipio")
plt.ylabel("Número de eventos")
plt.show()

In [ ]:
# C
free_by_category = (
    df.groupby("category")["is_free"]
      .agg(["sum","count"])
      .assign(percent_free=lambda x: (100*x["sum"]/x["count"]).round(1))
      .sort_values("percent_free", ascending=False)
)
free_by_category

## 7 · Break the scraper — 64–72 min
En el HTML cambia **una** aparición:

`class="price"` → `class="event-price"`

**PREDICE antes de ejecutar:** ¿error, dato perdido o resultado aparentemente normal?

Después: ¿puede un scraper producir resultados incorrectos sin que lo notemos?

Deshaz el cambio al terminar.

## 8 · Resultado → evidencia — 72–80 min
¿Qué porcentaje de eventos de **esta página** aparece como gratuito?

In [ ]:
pct_free = 100 * df["is_free"].mean()
round(pct_free, 1)

### Pregunta final
¿Podemos concluir:

> “X% de los eventos culturales en Puerto Rico son gratuitos”?

No. Nuestro universo es mucho más limitado: **esta página**, estas fechas, sus decisiones editoriales y nuestras reglas de extracción/limpieza.

### Exit ticket
1. Scraping es...
2. Una decisión que tomó el analista hoy fue...
3. Una forma en que un scraper puede engañarnos es...
4. Antes de creer una conclusión obtenida de datos web, preguntaría...